## Upload 20 Random Images to S3 with `laila.guarantee`

Create 20 random RGB images, upload them to an `S3Pool` inside `with laila.guarantee:`, and verify that the uploads completed when the context exits.


In [1]:
%load_ext autoreload
%autoreload 2

### Objective
Import dependencies, configure paths, and load the S3 pool class used by this example.


In [ ]:
import sys 
from pathlib import Path

import numpy as np
from PIL import Image

PROJECT_ROOT = Path("/home/ubuntu/laila")
PYTHON_ROOT = PROJECT_ROOT.parent
if str(PYTHON_ROOT) not in sys.path:
    sys.path.append(str(PYTHON_ROOT))

import laila
laila.read_args(PROJECT_ROOT / "vault" / "dev_secrets.toml")
from laila.pool import S3Pool


/home/ubuntu/laila/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Objective
Define the S3 pool helper and the constants for generating 20 deterministic random images.


In [3]:
POOL_NICKNAME = "guarantee_random_images_s3_pool"
IMAGE_COUNT = 20
IMAGE_SIZE = (64, 64)


def build_s3_pool() -> S3Pool:
    required = ["AWS_BUCKET_NAME", "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_REGION"]
    missing = [name for name in required if getattr(laila.args, name, None) is None]
    if missing:
        raise RuntimeError(f"Missing laila.args values: {', '.join(missing)}")

    return S3Pool(
        bucket_name=laila.args.AWS_BUCKET_NAME,
        access_key_id=laila.args.AWS_ACCESS_KEY_ID,
        secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
        region_name=laila.args.AWS_REGION,
        nickname=POOL_NICKNAME,
    )


### Objective
Create the S3 pool, generate all 20 random images first, then upload the full list inside `with laila.guarantee:` without manually waiting on the returned futures.


In [4]:
s3_pool = build_s3_pool()
laila.add_pool(s3_pool, pool_nickname=POOL_NICKNAME)

rng = np.random.default_rng(42)
image_entries = []

for i in range(IMAGE_COUNT):
    image_name = f"guarantee_image_{i}"
    image_array = rng.integers(0, 256, size=(*IMAGE_SIZE, 3), dtype=np.uint8)
    image = Image.fromarray(image_array, mode="RGB")
    entry = laila.constant(data=image, nickname=image_name)
    image_entries.append(entry)

with laila.guarantee:
    laila.memorize(image_entries, pool_nickname=POOL_NICKNAME)

print("Uploaded images:", len(image_entries))
print("First image id:", image_entries[0].global_id)
print("Last image id:", image_entries[-1].global_id)


Uploaded images: 20
First image id: LAILA:ENTRY:GLOBAL_ID:b51953c6-dc2e-585d-85de-d2b4d2f4f782
Last image id: LAILA:ENTRY:GLOBAL_ID:a7006ebf-ba50-546f-86ba-2f1545fbe8e7


### Objective
Remember a few uploaded images from S3 and verify that they come back as `PIL.Image.Image` objects.


In [5]:
sample_entries = image_entries[:3]
remember_future = laila.remember(
    [entry.global_id for entry in sample_entries],
    pool_nickname=POOL_NICKNAME,
)
remembered_images = remember_future.wait()

assert len(remembered_images) == len(sample_entries)
assert all(isinstance(entry.data, Image.Image) for entry in remembered_images)

[(entry.global_id, entry.data.size, entry.data.mode) for entry in remembered_images]


[('LAILA:ENTRY:GLOBAL_ID:b51953c6-dc2e-585d-85de-d2b4d2f4f782',
  (64, 64),
  'RGB'),
 ('LAILA:ENTRY:GLOBAL_ID:5fe06510-1fb7-55b3-8859-5e4400609d39',
  (64, 64),
  'RGB'),
 ('LAILA:ENTRY:GLOBAL_ID:e00081a8-bd1e-5290-ba2e-55ca16fd71d4',
  (64, 64),
  'RGB')]

### Objective
Clean up the S3 pool so the example does not leave uploaded images behind.


In [6]:
s3_pool.empty()
print("Emptied source S3 pool:", POOL_NICKNAME)


Emptied source S3 pool: guarantee_random_images_s3_pool
